# Archived-result reproduction
Data/results are not uploaded to GitHub. See [download and preparation instructions](../DATA.md). Raw source data alone do not contain the derived alignment results used here. Set `RAFTUP_EXPERIMENT_ROOT` to your revision archive root. These cells recompute plots and evaluations from saved results, not model training.


# Known overlap: 20%, 40%, 60%, 80%, 100%

从已有结果重绘；不训练、不重新配准。按顺序运行全部 cell。

输入根目录在第一个代码 cell 的 `ROOT`；所有新结果写入本文件夹的 `outputs/`，不覆盖原始结果。主要绘图代码已复制进本 notebook，可直接修改。

原始文件：
- `<revision_archive>/all_exp/04_plot_9cases_fig4_style_raftup_only.ipynb`

论文对应图片：`known_overlap.png`。

本版将 a–e 拆为五个独立绘图 cell。先运行共用数据 cell，再单独重跑需要修改的面板。字体设置集中在第二个代码 cell；每个面板导出 PNG（600 dpi）、PDF、SVG，外围留白为 0，输出位于 `outputs/01_known_overlap/panels/`。


In [ ]:
from pathlib import Path
import os, sys
os.environ.setdefault('MPLCONFIGDIR', '/tmp/raftup_reviewer_figures_mpl')
# Input project root. Change only this path when moving the project.
ROOT = Path(os.environ.get('RAFTUP_EXPERIMENT_ROOT', './revision_data')).expanduser().resolve()
NOTEBOOK_DIR = Path.cwd()
RESULTS_DIR = Path(os.environ.get('RAFTUP_OUTPUT_ROOT', './outputs')).expanduser().resolve()
OUT = RESULTS_DIR / '01_known_overlap' / 'panels'
OUT.mkdir(parents=True, exist_ok=True)
assert ROOT.is_dir(), f'Missing input project: {ROOT}'
print('Input project:', ROOT)
print('Outputs:', OUT)



In [ ]:
from pathlib import Path
import os, json, hashlib
os.environ.setdefault('MPLCONFIGDIR', '/tmp/raftup_overlap_plot_cache')
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from IPython.display import display

DATA = ROOT / '1_simulated_data_exp/151673_simulatedData'
RUNS = ROOT / 'all_exp/test'
OVERLAPS = [20, 40, 60, 80, 100]
MAX_EDGES = 160
SEED = 547
PALETTE = dict(zip(map(str, range(7)), ['#E9897E','#4FA3D1','#7BC8A4','#C27ACF','#F2B56B','#F3E37C','#C9B99A']))
PALETTE.update(dict(zip(['MPN','MPA','PV','V3'], ['#0072B2','#D55E00','#009E73','#CC79A7'])))
# Shared typography; change these settings to adjust all five panels.
FONT_SIZE = 12
LABEL_SIZE = 13
PANEL_SIZE = 18
DPI = 600
plt.rcParams.update({
    'font.family':'Arial', 'font.size':FONT_SIZE, 'axes.titlesize':LABEL_SIZE,
    'axes.labelsize':LABEL_SIZE, 'xtick.labelsize':11, 'ytick.labelsize':11,
    'legend.fontsize':11, 'pdf.fonttype':42, 'svg.fonttype':'none',
    'axes.spines.top':False, 'axes.spines.right':False,
    'savefig.bbox':'tight', 'savefig.pad_inches':0,
})


In [ ]:
def read_spatial(path):
    a = ad.read_h5ad(path, backed='r')
    result = dict(xy=np.asarray(a.obsm['spatial'],float)[:,:2].copy(),
                  label=a.obs.original_clusters.astype(str).to_numpy(),
                  ids=a.obs_names.to_numpy(),
                  spot=a.obs.SpotIndex.to_numpy() if 'SpotIndex' in a.obs else None)
    a.file.close()
    return result

def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(8*1024**2),b''):h.update(chunk)
    return h.hexdigest()

def verify_plan(name,a,b):
    path=RUNS/name/'alignment_matrix.npy';p=np.load(path)
    assert p.shape==(len(a['xy']),len(b['xy']))
    assert np.isfinite(p).all() and (p>=0).all() and p.sum()>0
    saved=json.loads((RUNS/name/'verification.json').read_text())
    rows=np.flatnonzero(p.max(1)>0);cols=p[rows].argmax(1)
    measured=dict(LAA=np.mean(a['label'][rows]==b['label'][cols]),coverage_A=(p.sum(1)>0).mean(),coverage_B=(p.sum(0)>0).mean())
    for k,v in measured.items():assert np.isclose(v,saved[k]),(name,k,v,saved[k])
    return p,dict(saved,alignment_sha256=sha(path))

def align_for_display(x,y,p):
    w=p/p.sum();mu=(w.sum(1)[:,None]*x).sum(0);nu=(w.sum(0)[:,None]*y).sum(0)
    u,_,vt=np.linalg.svd((x-mu).T@w@(y-nu))
    # Reflection permitted, consistent with the original Fig. 4 plotting code.
    return (x-mu)@(u@vt)+nu

def draw_mapping(ax,a,b,p,show_slice_labels=True):
    x=align_for_display(a['xy'],b['xy'],p);y=b['xy'].copy()
    both=np.vstack([x,y]);center=(both.min(0)+both.max(0))/2;scale=np.ptp(both,axis=0).max()
    x=(x-center)/scale;y=(y-center)/scale
    # Orthographic oblique projection of two separated planes; equal XY scale.
    def project(v,z):return np.column_stack([v[:,0]+.22*v[:,1],-.40*v[:,1]+z])
    xx=project(x,0);yy=project(y,.72)
    i=np.flatnonzero(p.max(1)>0);j=p[i].argmax(1)
    if len(i)>MAX_EDGES:
        keep=np.sort(np.random.default_rng(SEED).choice(len(i),MAX_EDGES,replace=False));i=i[keep];j=j[keep]
    ax.add_collection(LineCollection(np.stack([xx[i],yy[j]],axis=1),colors='#666666',linewidths=.25,alpha=.28,zorder=1))
    for v,labels in [(xx,a['label']),(yy,b['label'])]:
        ax.scatter(v[:,0],v[:,1],c=[PALETTE.get(s,'#999999') for s in labels],s=1.1,linewidths=0,zorder=2)
    bounds = np.vstack([xx, yy])
    lo, hi = bounds.min(0), bounds.max(0)
    ax.set(xlim=(lo[0]-.01,hi[0]+.01),ylim=(lo[1]-.01,hi[1]+.01))
    ax.set_aspect('equal');ax.axis('off')
    if show_slice_labels:
        ax.text(.02,.94,'B',transform=ax.transAxes,fontsize=6,color='.3')
        ax.text(.02,.13,'A',transform=ax.transAxes,fontsize=6,color='.3')

def save_panel(fig, name):
    # Crop to the visible artwork and labels, with no outer padding.
    for ext in ['pdf', 'svg', 'png']:
        fig.savefig(OUT / f'{name}.{ext}', dpi=DPI, bbox_inches='tight', pad_inches=0)
    fig.savefig(OUT / f'{name}_preview.png', dpi=160, bbox_inches='tight', pad_inches=0)
    plt.show()
    plt.close(fig)
    print('Saved:', OUT / f'{name}.png')


def metric_axes(letter):
    fig, ax = plt.subplots(figsize=(5.5, 3.8))
    ax.set(xticks=OVERLAPS, ylim=(0,1.05), xlabel='Nominal overlap (%)')
    ax.tick_params(labelsize=11)
    ax.grid(axis='y', color='.90', lw=.7)
    return fig, ax


## Extract and audit results
LAA is label agreement for positive row-top-1 matches and is **not exact matching accuracy**. GPR is read from the saved verification file (radius 300 for DLPFC; 30 for MERFISH, in native coordinate units).

Exact-match precision = correct positive row-top-1 pairs / all positive row-top-1 pairs. Exact-match recall = correct positive row-top-1 pairs / all ground-truth pairs. Ground-truth CSV axes and `SpotIndex` are checked before scoring. Overlap precision/recall compare predicted matched A spots against GT-overlapping A spots. Coverage is the fraction of spots with any positive transport mass, separately for A and B. A single run is shown per condition; no error bars or significance tests.


In [ ]:
summary=pd.read_csv(RUNS/'verification.csv')
records=[];overlap_data={};provenance=[]
a=read_spatial(DATA/'DLPFC_151673original_spotIndex.h5')
for percent in OVERLAPS:
    name=f'fig9_overlap_{percent}';pc='1.0' if percent==100 else '1'
    bp=DATA/f'DLPFC_151673_overlap={percent}%_pseudocount_{pc}_spotIndex.h5'
    gp=DATA/f'_overlap={percent}%mapping_matrix_ground_truth_spotIndex.csv'
    b=read_spatial(bp);p,saved=verify_plan(name,a,b)
    gdf=pd.read_csv(gp,index_col=0)
    assert np.array_equal(gdf.index.to_numpy(),np.arange(len(a['xy'])))
    assert np.array_equal(gdf.columns.astype(int),np.arange(len(b['xy'])))
    g=gdf.to_numpy();assert g.shape==p.shape and np.all((g>0).sum(0)==1)
    assert np.array_equal(a['spot'][g.argmax(0)],b['spot'])
    rows=np.flatnonzero(p.max(1)>0);cols=p[rows].argmax(1)
    correct=int((g[rows,cols]>0).sum());gtmask=(g>0).any(1);predmask=p.max(1)>0
    rec=dict(saved,overlap_percent=percent,n_A=len(a['xy']),n_B=len(b['xy']),n_GT_pairs=int((g>0).sum()),
             n_row_top1=len(rows),n_exact_correct=correct,exact_precision=correct/len(rows),exact_recall=correct/(g>0).sum(),
             overlap_A_precision=(gtmask&predmask).sum()/predmask.sum(),overlap_A_recall=(gtmask&predmask).sum()/gtmask.sum(),
             GT_coverage_A=float(gtmask.mean()),GT_coverage_B=float((g>0).any(0).mean()))
    records.append(rec);overlap_data[percent]=(a,b,p,g)
    provenance.append(dict(condition=name,matrix_sha256=saved['alignment_sha256'],GT_sha256=sha(gp),A_sha256=sha(DATA/'DLPFC_151673original_spotIndex.h5'),B_sha256=sha(bp)))
metrics=pd.DataFrame(records);metrics.to_csv(OUT/'overlap_metrics.csv',index=False)
summary.to_csv(OUT/'nine_cases_saved_metrics.csv',index=False)
(OUT/'input_provenance.json').write_text(json.dumps(provenance,indent=2))
display(metrics[['overlap_percent','LAA','GPR','coverage_A','coverage_B','exact_precision','exact_recall','overlap_A_precision','overlap_A_recall']])

## a — ground truth


In [ ]:
# Panel a: ground truth. Run after the shared data/metrics cells above.
fig, axes = plt.subplots(1, 5, figsize=(12, 3.2))
fig.subplots_adjust(left=.035, right=1, bottom=.03, top=.90, wspace=.08)
for column, (ax, percent) in enumerate(zip(axes, OVERLAPS)):
    a, b, p, g = overlap_data[percent]
    draw_mapping(ax, a, b, g, show_slice_labels=False)
    center_x = .035 + .965 * (column + .5) / 5
    fig.text(center_x, .94, f'{percent}% overlap', ha='center', fontsize=LABEL_SIZE)
fig.text(.006, .48, 'ground truth', rotation=90, fontsize=14, va='center')
save_panel(fig, 'panel_a_ground_truth')


## b — RAFT-UP


In [ ]:
# Panel b: RAFT-UP. Run after the shared data/metrics cells above.
fig, axes = plt.subplots(1, 5, figsize=(12, 3.2))
fig.subplots_adjust(left=.035, right=1, bottom=.03, top=.90, wspace=.08)
for column, (ax, percent) in enumerate(zip(axes, OVERLAPS)):
    a, b, p, g = overlap_data[percent]
    draw_mapping(ax, a, b, p, show_slice_labels=False)
    center_x = .035 + .965 * (column + .5) / 5
    fig.text(center_x, .94, f'{percent}% overlap', ha='center', fontsize=LABEL_SIZE)
    row = metrics.set_index('overlap_percent').loc[percent]
    ax.text(.5, -.07, f'LAA {row.LAA:.2f} | GPR {row.GPR:.2f}',
            ha='center', va='top', transform=ax.transAxes, fontsize=11)
fig.text(.006, .48, 'RAFT-UP', rotation=90, fontsize=14, va='center')
save_panel(fig, 'panel_b_raftup')


## c — LAA and GPR


In [ ]:
fig, ax = metric_axes('c')
x = metrics.overlap_percent
ax.plot(x, metrics.LAA, 'o-', color='#0072B2', ms=6, lw=1.8, label='LAA')
ax.plot(x, metrics.GPR, 's-', color='#D55E00', ms=6, lw=1.8, label='GPR')
ax.set_ylabel('Score')
ax.legend(frameon=False, loc='lower left')
save_panel(fig, 'panel_c_laa_gpr')


## d — Correct correspondence


In [ ]:
fig, ax = metric_axes('d')
x = metrics.overlap_percent
ax.plot(x, metrics.exact_precision, 'o-', color='#0072B2', ms=6, lw=1.8,
        label='# of correct alignment /\n# of total predicted alignment')
ax.plot(x, metrics.exact_recall, 's-', color='#D55E00', ms=6, lw=1.8,
        label='# of correct alignment /\n# of ground truth alignment')
ax.set_ylabel('Fraction')
ax.legend(frameon=False, loc='upper left', fontsize=11, labelspacing=1.0)
save_panel(fig, 'panel_d_correct_correspondence')


## e — Coverage


In [ ]:
fig, ax = metric_axes('e')
x = metrics.overlap_percent
ax.plot(x, metrics.coverage_A, 'o-', color='#0072B2', ms=6, lw=1.8, label='Predicted A')
ax.plot(x, metrics.coverage_B, 's-', color='#D55E00', ms=6, lw=1.8, label='Predicted B')
ax.plot(x, metrics.GT_coverage_A, '--', color='#0072B2', lw=1.5, label='ground truth A')
ax.plot(x, metrics.GT_coverage_B, '--', color='#D55E00', lw=1.5, label='ground truth B')
ax.set_ylabel('Coverage')
ax.legend(frameon=False, loc='lower right')
save_panel(fig, 'panel_e_coverage')
